In [1]:
# ==== HARD REPRO LOCK (put in the very first cell) ====
import os, random, numpy as np, torch
SEED = 2024  

# 1) env vars — 一定要在建立任何東西之前設定
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"  # CUDA matmul 決定論

# 2) 全域亂數源
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 3) cuDNN / 算法選擇（決定論）
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

# 4) DataLoader 專用 generator & worker seeding
g = torch.Generator().manual_seed(SEED)
def seed_worker(worker_id):
    base = SEED + worker_id
    np.random.seed(base)
    random.seed(base)
    torch.manual_seed(base)

print("✅ HARD REPRO LOCK enabled, SEED =", SEED)


✅ HARD REPRO LOCK enabled, SEED = 2024


In [75]:
# === 安裝/匯入（保留你原本的就好） ===
!pip -q install pydicom torch torchvision scikit-learn pandas matplotlib pillow

from pathlib import Path
import os, random, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
# 🔁 換成 ResNet50
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# === SPECT（Part 1）資料夾與檔名 ===
# 你把這個資料夾改成你存放 HW3 Part1 SPECT 的位置即可
SPECT_DATA_DIR = Path(r"C:\研究所\深度學習\hwk03_data")

# 作業提供的新訓練 CSV（含 Stage_New 六類）
SPECT_TRAIN_CSV = SPECT_DATA_DIR / "train_hwk02_new.csv"
# 測試 CSV
SPECT_TEST_CSV  = SPECT_DATA_DIR / "test.csv"

# （SPECT 不需要 DICOM；保留 DICOM 變數給 Part 2 MRI 再設定）
MRI_DATA_DIR  = Path(r"C:\研究所\深度學習\hwk03_data") 
MRI_DICOM_DIR = MRI_DATA_DIR / "DICOM"                        

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

# === 檔案存在性檢查 ===
print("[SPECT] 檔案檢查：", SPECT_TRAIN_CSV.exists(), SPECT_TEST_CSV.exists())
if not SPECT_TRAIN_CSV.exists():
    raise FileNotFoundError(f"找不到 {SPECT_TRAIN_CSV}，請確認資料夾與檔名。")

# === 快速讀一小段確認欄位（需包含 Stage_New 與 ID） ===
df_head = pd.read_csv(SPECT_TRAIN_CSV, nrows=5)
print("[SPECT] 訓練 CSV 欄位：", list(df_head.columns))
required_cols = {"Stage_New", "ID"}
missing = required_cols - set(df_head.columns)
if missing:
    raise ValueError(f"SPECT 訓練 CSV 缺少欄位：{missing}，請確認你拿到的檔案欄位名稱。")

# === 後續會用到的常數（Part 1） ===
NUM_CLASSES = 6
LABEL_COL   = "Stage_New"
ID_COL      = "ID"
OUTPUT_PROB_CSV = "ResNet50_6c.csv"

USE_CLASS_WEIGHTS    = True
USE_WEIGHTED_SAMPLER = True

print("[SPECT] NUM_CLASSES =", NUM_CLASSES, "| LABEL_COL =", LABEL_COL, "| ID_COL =", ID_COL)


device = cpu
[SPECT] 檔案檢查： True True
[SPECT] 訓練 CSV 欄位： ['ID', 'Age', 'Gender', 'FilePath', 'index', 'Stage_New']
[SPECT] NUM_CLASSES = 6 | LABEL_COL = Stage_New | ID_COL = ID



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [76]:
# === 讀取新 SPECT 訓練/測試資料，並拆分 train/valid ===
df_all = pd.read_csv(SPECT_TRAIN_CSV)
print(f"[SPECT] 原始資料筆數：{len(df_all)}")
print(df_all.head())

# 切分訓練與驗證
df_train, df_valid = train_test_split(
    df_all, test_size=0.2, random_state=42, stratify=df_all[LABEL_COL])

# 測試資料（沒有 Stage_New 標籤）
df_test = pd.read_csv(SPECT_TEST_CSV)
print(f"[SPECT] 訓練 {len(df_train)}, 驗證 {len(df_valid)}, 測試 {len(df_test)}")

# 把 Stage_New → Stage
def _map_new_stage_columns(df, label_col_new=LABEL_COL, target_col_for_old_dataset="Stage"):
    df = df.copy()
    if label_col_new in df.columns:
        df[target_col_for_old_dataset] = df[label_col_new].astype(int)
    else:
        raise ValueError(f"找不到新欄位 {label_col_new}，請確認你的 CSV。")
    return df

df_train_mapped = _map_new_stage_columns(df_train)
df_valid_mapped = _map_new_stage_columns(df_valid)

# 測試資料若沒有 Stage_New 就不需要 mapping
print("[SPECT] Columns after mapping:", df_train_mapped.columns.tolist()[:8])


[SPECT] 原始資料筆數：161
        ID  Age  Gender                 FilePath  index  Stage_New
0  A175204   69       0  /DICOM/A175204/00010018      7          2
1  A122221   56       0  /DICOM/A122221/00010034     15          1
2   A54671   82       0   /DICOM/A54671/00010021      8          1
3   A31117   71       1   /DICOM/A31117/00010022     10          2
4  A653195   68       0  /DICOM/A653195/00010016      7          2
[SPECT] 訓練 128, 驗證 33, 測試 40
[SPECT] Columns after mapping: ['ID', 'Age', 'Gender', 'FilePath', 'index', 'Stage_New', 'Stage']


In [77]:
from torchvision import transforms

IMG_SIZE = 224
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),   # ← 換掉 ImageNet 均值/方差
])

valid_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])


In [78]:
import numpy as np
import torch
from torch.utils.data import WeightedRandomSampler
import torch.nn as nn

# ---------------- class weights ----------------
def compute_class_weights_from_df(df, label_col=LABEL_COL, num_classes=NUM_CLASSES):
    counts = np.zeros(num_classes, dtype=np.float64)
    y = df[label_col].values.astype(int)
    for k in range(num_classes):
        counts[k] = (y == k).sum()
    counts = np.clip(counts, 1.0, None)
    inv = 1.0 / counts
    inv = inv * (num_classes / inv.sum())  # normalize to mean=1
    return torch.tensor(inv, dtype=torch.float32)

# ---------------- sampler ----------------
def make_weighted_sampler_from_df(df, label_col=LABEL_COL, num_classes=NUM_CLASSES):
    counts = np.zeros(num_classes, dtype=np.float64)
    y = df[label_col].values.astype(int)
    for k in range(num_classes):
        counts[k] = (y == k).sum()
    counts = np.clip(counts, 1.0, None)
    per_class_weight = 1.0 / counts
    sample_weights = per_class_weight[y]
    return WeightedRandomSampler(torch.tensor(sample_weights, dtype=torch.float32),
                                 num_samples=len(sample_weights), replacement=True)

# ---------------- 換最後分類層為 6 類 ----------------
def replace_classifier_to_num_classes(model: nn.Module, num_classes: int = NUM_CLASSES):
    # torchvision resnet 常見寫法
    if hasattr(model, "fc") and isinstance(model.fc, nn.Linear):
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model
    # 常見命名
    for attr in ["classifier", "head", "last_linear", "final_fc"]:
        if hasattr(model, attr) and isinstance(getattr(model, attr), nn.Linear):
            lin = getattr(model, attr)
            in_features = lin.in_features
            setattr(model, attr, nn.Linear(in_features, num_classes))
            return model
    # fallback：找最後一個 Linear 直接換
    last_linear = None
    for m in model.modules():
        if isinstance(m, nn.Linear):
            last_linear = m
    if last_linear is None:
        raise RuntimeError("找不到可替換的 Linear 分類頭，請手動調整。")
    in_features = last_linear.in_features
    new_linear = nn.Linear(in_features, num_classes)
    replaced = False
    for module in model.modules():
        for name, child in list(module.named_children()):
            if child is last_linear:
                setattr(module, name, new_linear)
                replaced = True
                break
        if replaced:
            break
    if not replaced:
        raise RuntimeError("替換分類層失敗，請檢查模型結構。")
    return model


In [79]:
from pathlib import Path

def ensure_test_paths(df_test, train_df=None, base_dir=SPECT_DATA_DIR):
    df = df_test.copy()

    has_resolved = "ResolvedPath" in df.columns
    has_file     = "FilePath" in df.columns

    if not (has_resolved or has_file):
        # 1) 盡量從訓練資料推根目錄
        root = None
        if train_df is not None:
            for col in ["ResolvedPath", "FilePath"]:
                if col in train_df.columns:
                    try:
                        first = str(train_df[col].dropna().iloc[0])
                        p = Path(first)
                        root = p if p.is_dir() else p.parent
                        break
                    except Exception:
                        pass
        # 2) 如果還沒推到，就用常見的兩個候選
        if root is None:
            for cand in [base_dir / "DICOM", base_dir]:
                if Path(cand).exists():
                    root = Path(cand)
                    break
        if root is None:
            root = Path(base_dir)

        # 3) 以 ID 組成路徑（多數作業資料的測試影像是以 ID 為資料夾名或檔名前綴）
        df["FilePath"] = df["ID"].astype(str).apply(lambda x: str(root / x))

    return df

# 套用修補
df_test = ensure_test_paths(df_test, train_df=df_train_mapped, base_dir=SPECT_DATA_DIR)

# 確認一下
print("[test path check] has ResolvedPath:", "ResolvedPath" in df_test.columns,
      "| has FilePath:", "FilePath" in df_test.columns)
print(df_test.head(3))


[test path check] has ResolvedPath: False | has FilePath: True
        ID  Disease 0  Disease 1   Disease                FilePath
0  1035303        NaN         NaN      NaN  \DICOM\A527914\1035303
1  1032110        NaN         NaN      NaN  \DICOM\A527914\1032110
2     3158        NaN         NaN      NaN     \DICOM\A527914\3158


In [89]:
# =========================================================
# 全新一格：修 FilePath 為絕對路徑 + 支援 DICOM 的 SpectDataset + 重建 DataLoader
# 需求：已經有 df_train_mapped, df_valid_mapped, df_test, SPECT_DATA_DIR, train_tfms, valid_tfms
# =========================================================
import os, numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ---------- 1) 補 FilePath：把相對路徑轉成以 SPECT_DATA_DIR 為根的「絕對路徑」 ----------
def _to_abs_under_root(p, root):
    p = str(p)
    p = p.lstrip("\\/")                  # 去掉開頭的斜線
    return str((root / p).resolve())

def ensure_abs_paths(df, root, col="FilePath"):
    if col in df.columns:
        out = df.copy()
        out[col] = out[col].astype(str).apply(lambda x: _to_abs_under_root(x, root))
        return out
    return df

# 若 train/valid/test 有 FilePath 欄，就轉成絕對路徑
try:
    df_train_mapped = ensure_abs_paths(df_train_mapped, SPECT_DATA_DIR, "FilePath")
except NameError:
    pass
try:
    df_valid_mapped = ensure_abs_paths(df_valid_mapped, SPECT_DATA_DIR, "FilePath")
except NameError:
    pass
df_test = ensure_abs_paths(df_test, SPECT_DATA_DIR, "FilePath")

print("[path] train_has_FilePath:", "FilePath" in getattr(df_train_mapped, "columns", []),
      "| valid_has_FilePath:", "FilePath" in getattr(df_valid_mapped, "columns", []),
      "| test_has_FilePath:",  "FilePath" in df_test.columns)

# ---------- 2) 支援 DICOM/影像/NPY 的 SpectDataset ----------
class SpectDataset(Dataset):
    def __init__(self, df, is_train=True, transform=None, use_three_slices=True,
                 age_mean=None, age_std=None, crop_size=128):
        self.df = df.reset_index(drop=True).copy()
        self.is_train = bool(is_train)
        self.transform = transform
        self.use_three_slices = bool(use_three_slices)
        self.crop_size = int(crop_size)

        # 年齡標準化參數
        if (age_mean is None) or (age_std is None):
            if "Age" in self.df.columns:
                age_series = pd.to_numeric(self.df["Age"], errors="coerce")
                self.age_mean = float(age_series.mean())
                self.age_std  = float(age_series.std() + 1e-6)
            else:
                self.age_mean, self.age_std = 0.0, 1.0
        else:
            self.age_mean, self.age_std = float(age_mean), float(age_std)

        # 欄位偵測
        self.has_stage = "Stage" in self.df.columns
        self.has_id    = "ID" in self.df.columns
        self.path_key  = "ResolvedPath" if "ResolvedPath" in self.df.columns else \
                         ("FilePath" if "FilePath" in self.df.columns else None)
        if self.path_key is None:
            raise KeyError("缺少路徑欄位（需要 ResolvedPath 或 FilePath）")
        self.slice_key = None
        for k in ["index", "Slice", "slice", "z"]:
            if k in self.df.columns:
                self.slice_key = k
                break

    # ---- 工具：讀單張（支援影像/DICOM/NPY）----
    def _load_gray(self, fpath: str) -> np.ndarray:
        p = Path(fpath)
        ext = p.suffix.lower()
        try:
            # 常見影像
            if ext in {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}:
                img = Image.open(p).convert("L")
                return np.array(img, dtype=np.float32)

            # NPY
            if ext == ".npy":
                arr = np.load(str(p))
                arr = np.asarray(arr)
                if arr.ndim == 3:        # (Z,H,W) → 取中間片
                    arr = arr[arr.shape[0]//2]
                elif arr.ndim != 2:
                    raise ValueError(f"Unsupported npy shape: {arr.shape}")
                return arr.astype(np.float32)

            # DICOM（或沒副檔名時嘗試 DICOM）
            if ext in {".dcm",".dicom"} or ext == "":
                try:
                    import pydicom
                    ds = pydicom.dcmread(str(p))
                    arr = ds.pixel_array.astype(np.float32)
                    return arr
                except Exception:
                    pass  # 不是 DICOM 就往下試 PIL

            # 最後再用 PIL 嘗試（有些檔沒有副檔名）
            img = Image.open(p).convert("L")
            return np.array(img, dtype=np.float32)
        except Exception:
            s = self.crop_size
            return np.zeros((s, s), dtype=np.float32)

    # ---- 工具：fp 是資料夾就遞迴找檔取第 z 張；是檔案就直接讀 ----
    def _read_frame_by_index(self, fp: str, z: int) -> np.ndarray:
        p = Path(fp)
        if p.is_dir():
            # 遞迴抓所有可讀副檔名
            pats = []
            for ext in ["*.png","*.jpg","*.jpeg","*.bmp","*.tif","*.tiff",
                        "*.webp","*.dcm","*.dicom","*.npy","*"]:
                pats += list(p.rglob(ext))
            files = [str(f) for f in sorted(set(f for f in pats if f.is_file()))]
            if not files:
                return np.zeros((self.crop_size, self.crop_size), dtype=np.float32)
            z = max(0, min(int(z), len(files)-1))
            return self._load_gray(files[z])
        else:
            return self._load_gray(str(p))

    def _three_slices_to_3ch(self, fp: str, z: int) -> np.ndarray:
        s0 = self._read_frame_by_index(fp, max(z-1,0))
        s1 = self._read_frame_by_index(fp, z)
        s2 = self._read_frame_by_index(fp, z+1)
        H, W = s1.shape
        def fit(x):
            y = np.zeros((H,W), dtype=np.float32)
            y[:min(H,x.shape[0]), :min(W,x.shape[1])] = x[:min(H,x.shape[0]), :min(W,x.shape[1])]
            return y
        s0, s2 = fit(s0), fit(s2)
        return np.stack([s0, s1, s2], axis=-1)

    def _center_crop(self, arr3: np.ndarray) -> np.ndarray:
        H, W, _ = arr3.shape
        s = self.crop_size
        cy, cx = H//2, W//2
        half = s//2
        y0, y1 = max(0,cy-half), min(H, cy+half + (s%2))
        x0, x1 = max(0,cx-half), min(W, cx+half + (s%2))
        out = arr3[y0:y1, x0:x1]
        # 邊緣 pad
        pad_h = max(0, s - out.shape[0])
        pad_w = max(0, s - out.shape[1])
        if pad_h or pad_w:
            out = np.pad(out, ((0,pad_h),(0,pad_w),(0,0)), mode="edge")
        return out[:s,:s,:]

    def _to_uint8(self, arr3: np.ndarray) -> np.ndarray:
        a = arr3.astype(np.float32)
        mn, mx = np.nanmin(a), np.nanmax(a)
        if not np.isfinite(mn) or not np.isfinite(mx) or mx <= mn:
            return np.zeros_like(a, dtype=np.uint8)
        a = (a - mn) / (mx - mn) * 255.0
        return a.clip(0,255).astype(np.uint8)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx: int):
        r = self.df.iloc[idx]

        # 路徑與切片
        fp = str(r.get(self.path_key))
        if pd.isna(fp): raise KeyError(f"{self.path_key} 為空 (row {idx})")
        if self.slice_key and not pd.isna(r.get(self.slice_key)):
            try: z = int(r[self.slice_key])
            except: z = 0
        else:
            z = 0

        # 讀圖 → 三切片/單切片 → 中心裁切 → uint8 → tensor
        if self.use_three_slices:
            arr3 = self._three_slices_to_3ch(fp, z)
        else:
            s = self._read_frame_by_index(fp, z)
            arr3 = np.stack([s,s,s], axis=-1)
        arr3 = self._center_crop(arr3)
        arr3 = self._to_uint8(arr3)
        pil  = Image.fromarray(arr3)
        x_img = self.transform(pil) if self.transform else transforms.ToTensor()(pil)

        # tabular
        def _gender_to_float(g):
            if isinstance(g, str):
                s = g.strip().upper()
                if s in {"M","MALE","1"}: return 1.0
                if s in {"F","FEMALE","0"}: return 0.0
            try:
                x = float(g)
                if np.isnan(x): return 0.0
                return 1.0 if x>=0.5 else 0.0
            except: return 0.0

        age_raw = r.get("Age", np.nan)
        try:    age = (float(age_raw) - self.age_mean)/(self.age_std if self.age_std!=0 else 1.0)
        except: age = 0.0
        gender = _gender_to_float(r.get("Gender", np.nan))
        x_tab = torch.tensor([age, gender], dtype=torch.float32)

        if self.is_train:
            if not self.has_stage: raise KeyError("訓練/驗證需要 'Stage' 欄位（0..5）")
            y = int(r["Stage"])
            return (x_img, x_tab), y
        else:
            id_out = r["ID"] if self.has_id else idx
            return (x_img, x_tab), id_out

from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np, torch

# datasets 已經建好：ds_train / ds_val / ds_test

# 用「訓練子集本身」的標籤做權重（避免越界）
y_sub = ds_train.df["Stage"].astype(int).values
cnt   = np.bincount(y_sub, minlength=6)
cnt[cnt==0] = 1
w_cls = 1.0 / np.sqrt(cnt)                 # 平滑過的反比權重，避免過度懲罰
w_smp = w_cls[y_sub]

sampler = WeightedRandomSampler(
    torch.tensor(w_smp, dtype=torch.float32),
    num_samples=len(y_sub),
    replacement=True
)

BATCH_SIZE = 16; NUM_WORKERS = 0; PIN_MEMORY = True
def seed_worker(i):
    np.random.seed(42+i); torch.manual_seed(42+i)
g = torch.Generator().manual_seed(42)

train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          worker_init_fn=seed_worker, generator=g, drop_last=False)
valid_loader = DataLoader(ds_val,   batch_size=32, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(ds_test,  batch_size=32, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# 小檢查：train/valid label 分佈
print("[train label hist]", np.bincount(ds_train.df["Stage"].astype(int).values, minlength=6))
print("[valid label hist]", np.bincount(ds_val.df["Stage"].astype(int).values,   minlength=6))


[path] train_has_FilePath: True | valid_has_FilePath: True | test_has_FilePath: True
[train label hist] [ 3 14 18 33 55  5]
[valid label hist] [ 1  3  5  9 14  1]


In [93]:
# === ResNet50 版本（相容舊參數：num_classes、use_pretrained、freeze_backbone）===
import torch
import torch.nn as nn
from torchvision import models

class ResNet50WithTabular(nn.Module):
    def __init__(
        self,
        num_classes: int = 1,           # 1=二元(輸出1個logit)；>1=多類別(輸出C個logits)
        tab_in_dim: int = 2,            # tabular維度；沒有tabular就設0
        use_pretrained: bool = True,    # 舊參數名
        pretrained: bool | None = None, # 新參數名；若同時填，pretrained優先
        freeze_backbone: bool = False
    ):
        super().__init__()
        if pretrained is None:
            pretrained = use_pretrained

        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.resnet50(weights=weights)

        # 拿掉原 fc，保留 2048 維特徵
        self.backbone.fc = nn.Identity()
        img_feat_dim = 2048

        # 可選 tabular 分支
        self.use_tab = (tab_in_dim is not None) and (tab_in_dim > 0)
        if self.use_tab:
            self.tab_mlp = nn.Sequential(
                nn.Linear(tab_in_dim, 32),
                nn.ReLU(inplace=True),
                nn.BatchNorm1d(32),
                nn.Dropout(0.10),
            )
            tab_feat_dim = 32
        else:
            tab_feat_dim = 0

        # 最終分類頭：2048(+32) -> hidden -> num_classes
        head_in = img_feat_dim + tab_feat_dim
        hidden = 256
        self.head = nn.Sequential(
            nn.Linear(head_in, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(hidden, num_classes if num_classes > 1 else 1)
        )
        self.num_classes = num_classes

        # 是否凍結 backbone
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x_img, x_tab=None):
        img_feat = self.backbone(x_img)  # (B,2048)
        if self.use_tab:
            if x_tab is None:
                raise ValueError("模型設定了 tab_in_dim>0，但 forward 沒有提供 x_tab。")
            tab_feat = self.tab_mlp(x_tab)  # (B,32)
            feat = torch.cat([img_feat, tab_feat], dim=1)
        else:
            feat = img_feat

        out = self.head(feat)
        if self.num_classes == 1:
            out = out.squeeze(1)  # (B,)：給 BCEWithLogitsLoss
        # 若 num_classes>1：維持 (B,C)，給 CrossEntropyLoss
        return out

In [94]:
# =========================
# Model + Loss + Optimizer
# =========================
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)

# ---- 建立模型 (使用你現成的 ResNet50WithTabular) ----
# 若你的類別名稱或參數不同，改這一行就好
# 先不吃 Age/Gender，排除 tabular 噪音
model = ResNet50WithTabular(
    num_classes=6,
    tab_in_dim=0,            # ← 先關掉 tabular
    use_pretrained=True,
    freeze_backbone=True
).to(device)

# 類別權重用「子集」計算 + smoothing
def _cls_w_from_df(df, col="Stage", C=6):
    import numpy as np, torch
    y = df[col].astype(int).values
    cnt = np.bincount(y, minlength=C).astype(np.float64)
    cnt[cnt==0] = 1
    w = 1.0 / np.sqrt(cnt)
    w = w * (C / w.sum())
    return torch.tensor(w, dtype=torch.float32)

class_weights = _cls_w_from_df(ds_train.df, "Stage", 6).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# 只訓練 head 起步
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=1e-3, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

# ---- 類別權重 CrossEntropyLoss ----
def _compute_class_weights_from_df(df, label_col="Stage", num_classes=6):
    import numpy as np, torch
    counts = np.zeros(num_classes, dtype=np.float64)
    y = df[label_col].values.astype(int)
    for k in range(num_classes):
        counts[k] = (y == k).sum()
    counts = np.clip(counts, 1.0, None)
    inv = 1.0 / counts
    inv = inv * (num_classes / inv.sum())  # 平均=1
    return torch.tensor(inv, dtype=torch.float32)

class_weights = _compute_class_weights_from_df(df_train_mapped, label_col="Stage", num_classes=6).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

# ---- 建立模型：先凍結骨幹，讓 head 先學到可用訊號 ----
model = ResNet50WithTabular(
    num_classes=6,
    tab_in_dim=2,            # 你有用 age/gender 就留 2；若想排除 tabular 測試，改成 0
    use_pretrained=True,
    freeze_backbone=True     # 先凍結，等下第4個 epoch 解凍
).to(device)

# ---- 類別權重 + 一點 label smoothing ----
class_weights = _compute_class_weights_from_df(df_train_mapped, label_col="Stage", num_classes=6).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

# ---- Optimizer：先只訓練 head（凍結狀態下 backbone 參數不會更新）----
LR_HEAD = 1e-3
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR_HEAD, weight_decay=1e-4)

from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)


device = cpu


In [95]:
# 以 ds_train 的 y 分佈來建立 sampler（長度 = len(ds_train)）
import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

# 取出訓練子集的標籤（注意：我們的 Dataset 內部 df 就是子集）
y_train_sub = ds_train.df["Stage"].astype(int).values
counts = np.bincount(y_train_sub, minlength=6)
counts[counts == 0] = 1  # 避免除 0
per_class_w = 1.0 / counts
sample_w = per_class_w[y_train_sub]
sampler_fixed = WeightedRandomSampler(
    torch.tensor(sample_w, dtype=torch.float32),
    num_samples=len(y_train_sub),
    replacement=True
)

# 重新建立 DataLoader（valid/test 不用動）
BATCH_SIZE = 16
NUM_WORKERS = 0
PIN_MEMORY = True
SEED = 42

def seed_worker(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed); torch.manual_seed(worker_seed)

g = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    ds_train, batch_size=BATCH_SIZE, sampler=sampler_fixed,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    worker_init_fn=seed_worker, generator=g, drop_last=False
)

print("[fixed] sampler length =", len(sample_w), "| ds_train length =", len(ds_train))


[fixed] sampler length = 128 | ds_train length = 128


In [96]:
# =====================
# Train / Validate Loop 
# =====================
import torch
import torch.nn.functional as F

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for (x_img, x_tab), y in loader:
        x_img = x_img.to(device, non_blocking=True)
        x_tab = x_tab.to(device, non_blocking=True)
        y     = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x_img, x_tab)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * x_img.size(0)
        pred = logits.argmax(1)
        correct += int((pred == y).sum().item())
        n += x_img.size(0)
    return total_loss / max(n,1), correct / max(n,1)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for (x_img, x_tab), y in loader:
        x_img = x_img.to(device, non_blocking=True)
        x_tab = x_tab.to(device, non_blocking=True)
        y     = y.to(device, non_blocking=True)

        logits = model(x_img, x_tab)
        loss = criterion(logits, y)

        total_loss += float(loss.item()) * x_img.size(0)
        pred = logits.argmax(1)
        correct += int((pred == y).sum().item())
        n += x_img.size(0)
    return total_loss / max(n,1), correct / max(n,1)

BEST_PATH = "best_spect6c.pth"
EPOCHS = 20                     # 多給點步數

for ep in range(1, EPOCHS+1):
    # 第4個 epoch 解凍 backbone，切換成分組學習率
    if ep == 4:
        for p in model.backbone.parameters(): p.requires_grad = True
        optimizer = torch.optim.AdamW([
            {"params": model.backbone.parameters(), "lr": 1e-4, "weight_decay": 1e-4},
            {"params": (p for n,p in model.named_parameters()
                        if n.startswith("head") or n.startswith("tab_mlp")),
             "lr": 5e-4, "weight_decay": 1e-4},
        ])
        print("[unfreeze] backbone + param-group LRs")

    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    va_loss, va_acc = evaluate(model, valid_loader, criterion, device)
    try: scheduler.step(float(va_acc))
    except: pass

    # 顯示 valid 預測直方圖（看是否還是單一類）
    with torch.no_grad():
        model.eval()
        hist = torch.zeros(6, dtype=torch.long)
        for (xi, xt), yv in valid_loader:
            pr = model(xi.to(device), (xt.to(device) if isinstance(xt, torch.Tensor) else None)).argmax(1).cpu()
            hist += torch.bincount(pr, minlength=6)

    print(f"Epoch {ep:02d}/{EPOCHS} | train acc={tr_acc:.4f} loss={tr_loss:.4f} | "
          f"valid acc={va_acc:.4f} loss={va_loss:.4f}")
    print(f"  [valid pred hist] {hist.tolist()}")

    torch.save({"model": model.state_dict(), "val_acc": float(va_acc)}, BEST_PATH)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        print(f"  [BEST] val_acc={best_val_acc:.4f} → saved {BEST_PATH}")

print("DONE. best_val_acc =", float(best_val_acc))


C:\Users\User\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 01/20 | train acc=0.2031 loss=1.5990 | valid acc=0.0303 loss=2.2108
  [valid pred hist] [33, 0, 0, 0, 0, 0]
Epoch 02/20 | train acc=0.1641 loss=1.6630 | valid acc=0.0303 loss=2.1916
  [valid pred hist] [0, 0, 0, 0, 0, 33]
Epoch 03/20 | train acc=0.1875 loss=1.4481 | valid acc=0.0303 loss=2.2620
  [valid pred hist] [33, 0, 0, 0, 0, 0]
[unfreeze] backbone + param-group LRs
Epoch 04/20 | train acc=0.2031 loss=1.4264 | valid acc=0.0303 loss=2.1972
  [valid pred hist] [26, 0, 0, 0, 0, 7]
Epoch 05/20 | train acc=0.1484 loss=1.5444 | valid acc=0.0303 loss=2.1725
  [valid pred hist] [2, 0, 0, 0, 0, 31]
Epoch 06/20 | train acc=0.1797 loss=1.4604 | valid acc=0.0303 loss=2.2617
  [valid pred hist] [1, 0, 0, 0, 0, 32]
Epoch 07/20 | train acc=0.1875 loss=1.3109 | valid acc=0.0303 loss=2.0951
  [valid pred hist] [16, 5, 0, 0, 0, 12]
Epoch 08/20 | train acc=0.2578 loss=1.1742 | valid acc=0.0606 loss=2.1119
  [valid pred hist] [16, 17, 0, 0, 0, 0]
  [BEST] val_acc=0.0606 → saved best_spect6c.pth

In [97]:
# ============================
# Inference & CSV (robust, no-NaN, clean IDs)
# ============================
import os
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np

BEST_PATH = "best_spect6c.pth"

# 載入最佳模型（若找不到就用記憶體中的模型）
if os.path.exists(BEST_PATH):
    ckpt = torch.load(BEST_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    print(f"[LOAD] {BEST_PATH} loaded (val_acc={ckpt.get('val_acc','?')})")
else:
    print(f"[WARN] {BEST_PATH} not found, using current in-memory weights.")

model.eval()

@torch.no_grad()
def predict_loader_robust(model, loader, device):
    ids_out, probs_out, pred_out = [], [], []
    for (x_img, x_tab), ids in loader:
        x_img = x_img.to(device, non_blocking=True)
        x_tab = x_tab.to(device, non_blocking=True)

        logits = model(x_img, x_tab).float()  # (B,6)
        # softmax -> 機率，並把 NaN/Inf 換成均勻分佈
        probs = F.softmax(logits, dim=1)
        probs = torch.nan_to_num(probs, nan=1.0/6, posinf=1.0/6, neginf=1.0/6)

        # 轉成 numpy
        probs_np = probs.detach().cpu().numpy()
        pred_np  = probs_np.argmax(1)

        # IDs: tensor/list 都轉乾淨的字串
        if torch.is_tensor(ids):
            ids_list = ids.detach().cpu().numpy().tolist()
        elif isinstance(ids, (list, tuple)):
            ids_list = list(ids)
        else:
            ids_list = [ids]
        ids_list = [str(int(i)) for i in ids_list]

        ids_out.extend(ids_list)
        probs_out.extend(probs_np.tolist())
        pred_out.extend(pred_np.tolist())
    return ids_out, probs_out, pred_out

all_ids, all_probs, all_pred = predict_loader_robust(model, test_loader, device)

# 組 CSV：ID, p0..p5, pred
cols = ["ID"] + [f"p{k}" for k in range(6)] + ["pred"]
rows = [[all_ids[i], *map(float, all_probs[i]), int(all_pred[i])] for i in range(len(all_ids))]
df_out = pd.DataFrame(rows, columns=cols)

out_name = "ResNet50_6c.csv"
df_out.to_csv(out_name, index=False)
print(f"Saved {out_name} {df_out.shape}")
df_out.head()


[LOAD] best_spect6c.pth loaded (val_acc=0.06060606060606061)
Saved ResNet50_6c.csv (40, 8)


,ID,p0,p1,p2,p3,p4,p5,pred
0,1035303,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0
1,1032110,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0
2,3158,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0
3,759590,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0
4,903423,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0
